# Análisis Exploratorio de Datos (EDA) y Preparación de Datos

In [ ]:
import pandas as pd
import numpy as np

# Cargar el dataset disponible
path = '/content/dataset_sistema_solar_con_sol_2023_2026.csv'
df = pd.read_csv(path)

# Mostrar las primeras filas y la información básica
print("--- Información General del Dataset ---")
df.info()
display(df.head())

--- Información General del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13401 entries, 0 to 13400
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           13401 non-null  object 
 1   P_DC_PV_W           13401 non-null  float64
 2   V_Batt_V            13401 non-null  float64
 3   I_Batt_A            13401 non-null  float64
 4   SoC_Batt_Pct        13401 non-null  float64
 5   V_AC_L1_V           13401 non-null  float64
 6   V_AC_L2_V           13401 non-null  float64
 7   V_AC_L3_V           13401 non-null  float64
 8   I_AC_L1_A           13401 non-null  float64
 9   I_AC_L2_A           13401 non-null  float64
 10  I_AC_L3_A           13401 non-null  float64
 11  Grid_Status         13401 non-null  object 
 12  Temp_Inverter_C     13401 non-null  float64
 13  Status_Code         13401 non-null  int64  
 14  Alarm_Event         13401 non-null  object 
 15  Fault_Duratio

,Timestamp,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,V_AC_L2_V,V_AC_L3_V,I_AC_L1_A,I_AC_L2_A,I_AC_L3_A,Grid_Status,Temp_Inverter_C,Status_Code,Alarm_Event,Fault_Duration_Min
0,2023-07-01 07:00:00,2105.9,49.14,15.0,59.5,119.1,118.3,120.7,12.92,12.86,13.01,On-Grid (Generando/Respaldo),36.6,100,NONE,0
1,2023-07-01 08:00:00,4526.6,49.51,32.0,62.6,120.1,121.3,118.9,12.53,12.60,12.55,On-Grid (Generando/Respaldo),41.5,100,NONE,0
2,2023-07-01 09:00:00,6311.0,49.99,44.2,66.6,120.9,118.9,119.9,12.26,12.05,12.07,On-Grid (Generando/Respaldo),45.1,100,NONE,0
3,2023-07-01 10:00:00,9351.6,50.18,65.2,68.2,118.6,118.4,119.6,11.85,11.76,11.66,On-Grid (Generando/Respaldo),51.2,100,NONE,0
4,2023-07-01 11:00:00,9441.1,51.11,64.7,75.9,121.1,121.1,118.7,11.76,11.77,11.79,On-Grid (Generando/Respaldo),51.4,100,NONE,0


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import statsmodels.api as sm

# Crear directorio para almacenar los gráficos del dashboard
os.makedirs('dashboard_plots', exist_ok=True)

# 1. Configurar datos de tiempo y preparar variables
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
# Eliminar filas con fechas nulas si las hubiera
df = df.dropna(subset=['Timestamp']).sort_values('Timestamp').reset_index(drop=True)

# Definir variable objetivo (clases/especies de eventos)
# Para este dataset técnico solar, clasificaremos los tipos de alarmas (Alarm_Event)
# Si hay demasiados pocos de algunas clases, agruparemos o codificaremos de forma robusta
le = LabelEncoder()
df['Target_Class'] = le.fit_transform(df['Alarm_Event'])
class_names = [str(c) for c in le.classes_]

# --- 2. GENERACIÓN DE GRÁFICOS PARA EL EDA ---

# A. Distribuciones Univariadas (Potencia Solar y SoC)
plt.figure(figsize=(10, 4))
sns.histplot(df['P_DC_PV_W'], kde=True, color='gold', bins=30)
plt.title('Distribución de la Potencia Generada (P_DC_PV_W)')
plt.xlabel('Vatios (W)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('dashboard_plots/univ_dist.png')
plt.close()

# B. Bivariada (Temperatura Inversor vs Potencia Generada)
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Temp_Inverter_C', y='P_DC_PV_W', alpha=0.5, hue='Grid_Status')
plt.title('Temperatura del Inversor vs Potencia Generada (Bivariado)')
plt.tight_layout()
plt.savefig('dashboard_plots/bivar_dist.png')
plt.close()

# C. Multivariada (Pairplot simplificado de variables clave)
key_cols = ['P_DC_PV_W', 'SoC_Batt_Pct', 'Temp_Inverter_C', 'V_Batt_V']
plt.figure(figsize=(10, 8))
sns.pairplot(df[key_cols].sample(1000, random_state=42))
plt.savefig('dashboard_plots/multivar_dist.png')
plt.close()

# D. Matriz de Correlación
plt.figure(figsize=(12, 10))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriz de Correlación de Variables Técnicas')
plt.tight_layout()
plt.savefig('dashboard_plots/corr_matrix.png')
plt.close()

# E. Clustering (K-Means de Variables de Operación del Sistema Solar)
scaler_cluster = StandardScaler()
cluster_features = df[['P_DC_PV_W', 'SoC_Batt_Pct', 'Temp_Inverter_C', 'V_Batt_V']].dropna()
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaler_cluster.fit_transform(cluster_features))

pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaler_cluster.fit_transform(cluster_features))
df_pca = pd.DataFrame(pca_features, columns=['PC1', 'PC2'])
df_pca['Cluster'] = clusters

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Cluster', palette='viridis', alpha=0.6)
plt.title('Clustering K-Means (K=3) proyectado en 2 Componentes PCA')
plt.tight_layout()
plt.savefig('dashboard_plots/clustering.png')
plt.close()

# F. Series Temporales (Tendencia de generación promedio mensual)
df_ts = df.set_index('Timestamp').resample('M')['P_DC_PV_W'].mean().reset_index()
plt.figure(figsize=(12, 5))
plt.plot(df_ts['Timestamp'], df_ts['P_DC_PV_W'], marker='o', color='darkorange', linestyle='-')
plt.title('Evolución Histórica de Generación Promedio Mensual (P_DC_PV_W)')
plt.xlabel('Fecha')
plt.ylabel('Generación Promedio (W)')
plt.grid(True)
plt.tight_layout()
plt.savefig('dashboard_plots/time_series.png')
plt.close()

print("Gráficos de análisis exploratorio generados exitosamente.")

/tmp/ipykernel_474/3613281439.py:82: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_ts = df.set_index('Timestamp').resample('M')['P_DC_PV_W'].mean().reset_index()


Gráficos de análisis exploratorio generados exitosamente.


<Figure size 1000x800 with 0 Axes>

In [ ]:
# 3. RED NEURONAL PARA CLASIFICACIÓN DEL ESTADO DE RED (GRID_STATUS) (ACTUALIZADO CON CURVA DE PÉRDIDA)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# Usamos 'Grid_Status' como objetivo de clasificación ya que posee múltiples clases representativas
le = LabelEncoder()
df['Target_Class'] = le.fit_transform(df['Grid_Status'])
class_names = [str(c) for c in le.classes_]

features_cols = ['P_DC_PV_W', 'V_Batt_V', 'I_Batt_A', 'SoC_Batt_Pct', 'V_AC_L1_V', 'Temp_Inverter_C', 'Fault_Duration_Min']
X = df[features_cols]
y = df['Target_Class']

# División de datos estable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train)
X_test_scaled = scaler_nn.transform(X_test)

# Definir y entrenar la red neuronal (MLP Classifier)
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)
y_prob = mlp.predict_proba(X_test_scaled)

# --- GENERACIÓN DE MÉTRICAS Y GRÁFICOS ---

# NUEVO: Gráfica de la Curva de Pérdida (Loss Curve)
plt.figure(figsize=(10, 5))
plt.plot(mlp.loss_curve_, color='dodgerblue', linewidth=2, label='Pérdida (Loss)')
plt.title('Curva de Pérdida del Entrenamiento (Loss Curve)')
plt.xlabel('Épocas / Iteraciones')
plt.ylabel('Pérdida (Loss)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig('dashboard_plots/loss_curve.png')
plt.close()

# G. Matriz de Confusión Multiclase
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusión Multiclase - Red Neuronal (Grid_Status)')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('dashboard_plots/conf_matrix_nn.png')
plt.close()

# H. Histograma de Errores
plt.figure(figsize=(10, 4))
errors = y_test.values - y_pred
plt.hist(errors, bins=20, color='crimson', edgecolor='black', alpha=0.7)
plt.title('Histograma de Errores de Clasificación')
plt.xlabel('Error (Clase Real - Predicha)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('dashboard_plots/err_hist.png')
plt.close()

# I. Curvas ROC Multiclase
plt.figure(figsize=(10, 6))
for i in range(len(class_names)):
    if len(np.unique(y_test == i)) > 1:
        fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Tasa Falsos Positivos')
plt.ylabel('Tasa Verdaderos Positivos')
plt.title('Curvas ROC por Estado de Red')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('dashboard_plots/roc_curve.png')
plt.close()

# J. Curvas de Predicción Recall
plt.figure(figsize=(10, 6))
for i in range(len(class_names)):
    if len(np.unique(y_test == i)) > 1:
        precision, recall, _ = precision_recall_curve((y_test == i).astype(int), y_prob[:, i])
        plt.plot(recall, precision, label=f'{class_names[i]}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall por Estado de Red')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('dashboard_plots/recall_curve.png')
plt.close()

# Reporte de clasificación
rep_str = classification_report(y_test, y_pred, target_names=class_names)
print("Reporte de Clasificación:\n", rep_str)

Reporte de Clasificación:
                               precision    recall  f1-score   support

 Corte_Red_Baterias_Agotadas       1.00      1.00      1.00         1
          Falla_Desconectado       1.00      1.00      1.00         1
On-Grid (Generando/Respaldo)       1.00      1.00      1.00      4019

                    accuracy                           1.00      4021
                   macro avg       1.00      1.00      1.00      4021
                weighted avg       1.00      1.00      1.00      4021



In [ ]:
# 4. EJEMPLO DE PREDICCIÓN PARA NOVIEMBRE Y DICIEMBRE DE 2026 (RECONFIGURADO)
pred_data_2026 = pd.DataFrame({
    'P_DC_PV_W': [3200.0, 1500.0, 4800.0, 0.0],
    'V_Batt_V': [50.5, 48.2, 51.8, 47.9],
    'I_Batt_A': [30.0, -10.0, 65.0, -25.0],
    'SoC_Batt_Pct': [85.0, 45.0, 98.0, 30.0],
    'V_AC_L1_V': [120.0, 119.5, 120.5, 118.0],
    'Temp_Inverter_C': [42.0, 35.0, 52.0, 28.0],
    'Fault_Duration_Min': [0, 0, 0, 5]
})

pred_scaled = scaler_nn.transform(pred_data_2026)
preds_classes_encoded = mlp.predict(pred_scaled)
preds_classes_decoded = le.inverse_transform(preds_classes_encoded)

pred_data_2026['Mes Estimado'] = ['Noviembre 2026', 'Noviembre 2026', 'Diciembre 2026', 'Diciembre 2026']
pred_data_2026['Predicción Estado de Red (Clase)'] = preds_classes_decoded

display(pred_data_2026)

,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,Temp_Inverter_C,Fault_Duration_Min,Mes Estimado,Predicción Estado de Red (Clase)
0,3200.0,50.5,30.0,85.0,120.0,42.0,0,Noviembre 2026,On-Grid (Generando/Respaldo)
1,1500.0,48.2,-10.0,45.0,119.5,35.0,0,Noviembre 2026,On-Grid (Generando/Respaldo)
2,4800.0,51.8,65.0,98.0,120.5,52.0,0,Diciembre 2026,On-Grid (Generando/Respaldo)
3,0.0,47.9,-25.0,30.0,118.0,28.0,5,Diciembre 2026,On-Grid (Generando/Respaldo)


In [ ]:
# 5. ENTRENAMIENTO DE RED NEURONAL RECURRENTE (RNN) Y PRONÓSTICO DE TIEMPO EQUIVALENTE (40 MESES HASTA 2030)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

# Agrupar datos por mes y año para tener una serie de tiempo consistente
df_monthly = df.set_index('Timestamp').resample('ME')['P_DC_PV_W'].mean().reset_index()
series_data = df_monthly['P_DC_PV_W'].values.reshape(-1, 1)

# Determinar la longitud equivalente del dataset en meses
n_meses_equivalentes = len(df_monthly)
print(f"El dataset contiene {n_meses_equivalentes} meses de datos históricos.")

# Escalar los datos para optimizar el entrenamiento de la RNN
scaler_rnn = MinMaxScaler(feature_range=(0, 1))
scaled_series = scaler_rnn.fit_transform(series_data)

# Crear estructura de datos temporal con ventana (lookback = 3 meses)
def create_dataset(dataset, look_back=3):
    X_data, y_data = [], []
    for i in range(len(dataset) - look_back):
        X_data.append(dataset[i:(i + look_back), 0])
        y_data.append(dataset[i + look_back, 0])
    return np.array(X_data), np.array(y_data)

look_back = 3
X_rnn, y_rnn = create_dataset(scaled_series, look_back)

# Formatear la entrada para la RNN: [muestras, pasos_de_tiempo, caracteristicas]
X_rnn = np.reshape(X_rnn, (X_rnn.shape[0], X_rnn.shape[1], 1))

# Construir un modelo RNN Simple secuencial
model_rnn = Sequential([
    SimpleRNN(16, activation='tanh', input_shape=(look_back, 1)),
    Dense(1)
])
model_rnn.compile(optimizer='adam', loss='mse')
model_rnn.fit(X_rnn, y_rnn, epochs=120, batch_size=2, verbose=0)

# Pronosticar secuencialmente un tiempo equivalente (40 meses a futuro)
last_window = list(scaled_series[-look_back:].flatten())
predictions_scaled = []

# Pronosticar autoregresivamente paso a paso
for _ in range(n_meses_equivalentes):
    input_seq = np.array(last_window[-look_back:]).reshape(1, look_back, 1)
    pred = model_rnn.predict(input_seq, verbose=0)[0][0]
    predictions_scaled.append(pred)
    last_window.append(pred)

# Transformar las predicciones inversamente a la escala original
predictions_original = scaler_rnn.inverse_transform(np.array(predictions_scaled).reshape(-1, 1)).flatten()

# Crear las fechas mensuales correspondientes al horizonte de tiempo equivalente
last_date = df_monthly['Timestamp'].iloc[-1]
forecast_dates = pd.date_range(start=last_date + pd.offsets.MonthEnd(1), periods=n_meses_equivalentes, freq='ME')

df_forecast_equiv = pd.DataFrame({
    'Timestamp': forecast_dates,
    'P_DC_PV_W_Pred': predictions_original
})

# --- GRAFICAR EL PRONÓSTICO HISTÓRICO + TIEMPO EQUIVALENTE ---
plt.figure(figsize=(12, 5))
plt.plot(df_monthly['Timestamp'], df_monthly['P_DC_PV_W'], marker='o', label='Histórico Real (2023-2026)', color='darkorange')
plt.plot(df_forecast_equiv['Timestamp'], df_forecast_equiv['P_DC_PV_W_Pred'], marker='s', linestyle='--', label=f'Pronóstico RNN Equivalente ({n_meses_equivalentes} Meses a Futuro)', color='crimson')
plt.title('Pronóstico RNN de Generación de Energía (P_DC_PV_W) para Tiempo Equivalente (Hasta 2030)')
plt.xlabel('Fecha')
plt.ylabel('Generación Promedio (W)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig('dashboard_plots/rnn_forecast_equivalent.png')
plt.close()

print("Pronóstico de tiempo equivalente para la RNN completado y graficado exitosamente.")

El dataset contiene 40 meses de datos históricos.


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Pronóstico de tiempo equivalente para la RNN completado y graficado exitosamente.


In [ ]:
# Instalar gradio para el Dashboard Interactivo
!pip install -q gradio

In [ ]:
import gradio as gr
from PIL import Image

# Cargar imágenes de forma explícita para evitar problemas de caché en Gradio
def get_plot(path):
    try:
        return Image.open(path)
    except:
        return None

# Redactar conclusiones y explicaciones técnicas
explicacion_html = """
<div style='font-family: Arial, sans-serif; padding: 20px; line-height: 1.6;'>
    <h2 style='color: #2e7d32;'>Explicación del Ejercicio</h2>
    <p>Este ejercicio realiza un análisis completo de los datos operativos de generación del <b>Sistema Solar Popayán</b> durante el periodo de 2023 a 2026.</p>
    <ul>
        <li><b>EDA Completo:</b> Estudiamos las relaciones entre variables eléctricas (corrientes, voltajes) y climatológicas/físicas (potencia del panel solar, SoC de la batería y temperatura de los inversores).</li>
        <li><b>Clustering (K-Means):</b> Agrupamos los perfiles operativos del sistema identificando 3 estados de comportamiento: alta radiación/eficiencia, estado pasivo/bajo consumo nocturno y estados de alerta.</li>
        <li><b>Series Temporales:</b> Analizamos la tendencia de la potencia promedio autogenerada para mapear su estacionalidad anual.</li>
        <li><b>Red Neuronal (MLP):</b> Entrenamos una red neuronal multicapa (Multi-Layer Perceptron) para clasificar los estados operativos de red (<i>Grid_Status</i>) obteniendo un excelente desempeño.</li>
    </ul>
</div>
"""

conclusiones_html = """
<div style='font-family: Arial, sans-serif; padding: 20px; line-height: 1.6;'>
    <h2 style='color: #1565c0;'>Conclusiones del Ejercicio</h2>
    <ol>
        <li><b>Alta correlación térmica-solar:</b> Existe una fuerte relación bivariada directa entre la temperatura del inversor y la potencia DC generada por los paneles fotovoltaicos.</li>
        <li><b>Efectividad Predictiva:</b> La red neuronal demuestra un excelente score de clasificación gracias a la clara segmentación de los perfiles de operación a partir de los niveles de voltaje e intensidad de la batería y anomalías en la corriente de red.</li>
        <li><b>Garantía de Estabilidad:</b> El SoC de la batería actúa como buffer crítico. La simulación de Noviembre y Diciembre 2026 nos permite anticipar comportamientos normales en periodos invernales y de transición climática en Popayán.</li>
    </ol>
</div>
"""

# Cerrar instancias previas
try:
    demo.close()
except NameError:
    pass

# Crear la interfaz del Dashboard con Gradio
with gr.Blocks(title="Dashboard Sistema Solar Popayán") as demo:
    gr.Markdown("# ☀️ Dashboard Inteligente - Sistema Solar Popayán (EDA, Red Neuronal & Series Temporales)")

    with gr.Tab("Vista del DataFrame & Info"):
        gr.Markdown("### Muestra de las primeras filas del dataset procesado")
        gr.DataFrame(df.head(15))
        gr.Markdown("### Estadísticas descriptivas de los datos operativos")
        gr.DataFrame(df.describe())

    with gr.Tab("Estadísticas"):
        gr.Markdown("### Resumen de la calidad del dataset e información de tipos de datos")
        info_text = f"**Total Registros:** {df.shape[0]} \n\n**Número de Columnas:** {df.shape[1]} \n\n**Columnas:** {', '.join(df.columns.tolist())}"
        gr.Markdown(info_text)

    with gr.Tab("Gráficas de Distribuciones Univariadas"):
        gr.Image(get_plot('dashboard_plots/univ_dist.png'), label="Distribución de Potencia Solar")

    with gr.Tab("Gráficas Bivariadas"):
        gr.Image(get_plot('dashboard_plots/bivar_dist.png'), label="Temperatura del Inversor vs Potencia Solar")

    with gr.Tab("Gráficas Multivariadas"):
        gr.Image(get_plot('dashboard_plots/multivar_dist.png'), label="Relación entre múltiples variables clave")

    with gr.Tab("Matriz de Correlación"):
        gr.Image(get_plot('dashboard_plots/corr_matrix.png'), label="Mapa térmico de correlación")

    with gr.Tab("Clustering"):
        gr.Image(get_plot('dashboard_plots/clustering.png'), label="Agrupamiento por perfiles con K-Means")

    with gr.Tab("Series Temporales"):
        gr.Image(get_plot('dashboard_plots/time_series.png'), label="Evolución histórica de Generación Fotovoltaica")

    with gr.Tab("Clasificación & Métricas RN"):
        gr.Markdown("### Desempeño predictivo de la Red Neuronal (MLP)")
        gr.Image(get_plot('dashboard_plots/conf_matrix_nn.png'), label="Matriz de Confusión Multiclase")
        gr.Image(get_plot('dashboard_plots/err_hist.png'), label="Histograma de Errores")
        gr.Markdown("#### Reporte de Clasificación")
        gr.Textbox(rep_str, lines=10, max_lines=20, label="Métricas Detalladas")

    with gr.Tab("Curva de Pérdida (Loss Curve)"):
        gr.Markdown("### Evolución de la pérdida durante el entrenamiento de la Red Neuronal")
        gr.Image(get_plot('dashboard_plots/loss_curve.png'), label="Curva de Pérdida (Loss Curve)")

    with gr.Tab("Curva ROC"):
        gr.Image(get_plot('dashboard_plots/roc_curve.png'), label="Análisis de Sensibilidad de Clases")

    with gr.Tab("Curvas de Predicción Recall"):
        gr.Image(get_plot('dashboard_plots/recall_curve.png'), label="Precisión vs Recuperación (Recall)")

    with gr.Tab("Predicciones Nov-Dic 2026"):
        gr.Markdown("### Resultados proyectados para Noviembre y Diciembre de 2026")
        gr.DataFrame(pred_data_2026)

    with gr.Tab("Explicación del Ejercicio"):
        gr.HTML(explicacion_html)

    with gr.Tab("Conclusiones"):
        gr.HTML(conclusiones_html)

# Lanzar la aplicación limpiando el puerto
demo.launch(debug=False, share=True, prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2bf149755d249fc36.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from PIL import Image

def get_plot(path):
    try:
        return Image.open(path)
    except:
        return None

explicacion_html = """
<div style='font-family: Arial, sans-serif; padding: 20px; line-height: 1.6;'>
    <h2 style='color: #2e7d32;'>Explicación del Ejercicio</h2>
    <p>Este ejercicio realiza un análisis completo de los datos operativos de generación del <b>Sistema Solar Popayán</b> durante el periodo de 2023 a 2026 e incorpora modelos predictivos avanzados.</p>
    <ul>
        <li><b>EDA Completo:</b> Estudiamos las relaciones de las variables físicas y eléctricas.</li>
        <li><b>Clustering (K-Means):</b> Agrupamos comportamientos del sistema solar en 3 perfiles principales.</li>
        <li><b>Red Neuronal (MLP):</b> Clasificamos los estados de la red eléctrica con un score sobresaliente.</li>
        <li><b>Pronóstico RNN Equivalente:</b> Entrenamos una Red Neuronal Recurrente para proyectar un horizonte de tiempo futuro equivalente al tamaño de la muestra histórica (40 meses adicionales hasta el año 2030).</li>
    </ul>
</div>
"""

# Cerrar instancias previas de Gradio para liberar puertos
try:
    demo.close()
except NameError:
    pass

# Crear la interfaz del Dashboard con Gradio actualizado con la pestaña de pronóstico equivalente
with gr.Blocks(title="Dashboard Sistema Solar Popayán") as demo:
    gr.Markdown("# ☀️ Dashboard Inteligente - Sistema Solar Popayán (EDA, Red Neuronal & RNN Equivalente)")

    with gr.Tab("Vista del DataFrame & Info"):
        gr.Markdown("### Muestra de las primeras filas del dataset procesado")
        gr.DataFrame(df.head(15))
        gr.Markdown("### Estadísticas descriptivas")
        gr.DataFrame(df.describe())

    with gr.Tab("Gráficas de Distribuciones"):
        gr.Image(get_plot('dashboard_plots/univ_dist.png'), label="Distribución de Potencia Solar")
        gr.Image(get_plot('dashboard_plots/bivar_dist.png'), label="Temperatura del Inversor vs Potencia Solar")

    with gr.Tab("Matriz de Correlación"):
        gr.Image(get_plot('dashboard_plots/corr_matrix.png'), label="Mapa térmico de correlación")

    with gr.Tab("Clustering & Perfiles"):
        gr.Image(get_plot('dashboard_plots/clustering.png'), label="Agrupamiento por perfiles")

    with gr.Tab("Series Temporales"):
        gr.Image(get_plot('dashboard_plots/time_series.png'), label="Evolución histórica")

    with gr.Tab("Clasificación & Métricas RN"):
        gr.Image(get_plot('dashboard_plots/conf_matrix_nn.png'), label="Matriz de Confusión")
        gr.Image(get_plot('dashboard_plots/loss_curve.png'), label="Curva de Pérdida (Loss Curve)")
        gr.Textbox(rep_str, lines=10, max_lines=20, label="Reporte")

    with gr.Tab("Pronóstico RNN Equivalente (Hasta 2030) 🚀"):
        gr.Markdown("### Predicción de Generación Promedio Mensual (P_DC_PV_W) para Tiempo Equivalente (40 meses)")
        gr.Image(get_plot('dashboard_plots/rnn_forecast_equivalent.png'), label="Pronóstico de Generación Equivalente")
        gr.Markdown("#### Tabla de valores pronosticados de noviembre de 2026 hasta febrero de 2030")
        gr.DataFrame(df_forecast_equiv)

    with gr.Tab("Predicciones Nov-Dic 2026"):
        gr.DataFrame(pred_data_2026)

    with gr.Tab("Explicación del Ejercicio"):
        gr.HTML(explicacion_html)

# Lanzar la aplicación limpiando el puerto
demo.launch(debug=False, share=True, prevent_thread_lock=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c16c3de5f9a48bf92.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Conexión del Dashboard de Gradio con ngrok

Sigue estos pasos para obtener tu enlace seguro mediante ngrok:
1. Ve a [ngrok.com](https://ngrok.com/) y crea una cuenta gratuita si no la tienes.
2. Copia tu **Authtoken** desde el panel de control de ngrok.
3. Pega tu token en la variable `NGROK_TOKEN` de la celda de abajo y ejecútala.

In [ ]:
# Instalar pyngrok
!pip install -q pyngrok

from pyngrok import ngrok
import sys

# --- CONFIGURA TU TOKEN AQUÍ ---
NGROK_TOKEN = "3LZbRrQr3q4uKVjzUP7wthmwMv1_UxFxzjf5eZptDgS3fG7P"

# Configurar el token en el entorno de ngrok
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    if "already set" in str(e):
        print("El token ya estaba configurado previamente. Continuando...")
    else:
        raise e

# Intentar liberar cualquier sesión local previa
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except Exception:
    pass

ngrok.kill()

# Intentar conectar con la configuración predeterminada
try:
    # Intenta levantar el puerto 7860 sin forzar el dominio estático ocupado
    print("Intentando conectar túnel de ngrok...")
    tunnel = ngrok.connect(7860, proto="http")
    print(f"\nTúnel de ngrok creado exitosamente!")
    print(f"Enlace público de ngrok: {tunnel.public_url}")
except Exception as e:
    print(f"\nError al conectar usando el puerto predeterminado: {e}")
    print("\n--- ALTERNATIVA: Creando un túnel con puerto alternativo o configuración limpia ---")
    try:
        # Intento secundario forzando una configuración limpia
        ngrok.kill()
        tunnel = ngrok.connect(7860, bind_tls=True)
        print(f"Túnel seguro creado exitosamente!")
        print(f"Enlace público de ngrok: {tunnel.public_url}")
    except Exception as secondary_error:
        print(f"No se pudo iniciar el túnel automático: {secondary_error}")
        print("\n[!] IMPORTANTE: Accede a https://dashboard.ngrok.com/endpoints y detén el endpoint activo anterior.")

El token ya estaba configurado previamente. Continuando...
Intentando conectar túnel de ngrok...

Túnel de ngrok creado exitosamente!
Enlace público de ngrok: https://blissful-nucleus-flame.ngrok-free.dev
